WE will build a sample neural network 

We will learn how to build a simple neural network pipeline


In [6]:
import pandas as pd
import torch 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [36]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [ ]:
'''
Code flow

1. load the dataset
2. basic programming
3. training process
   a. create the model
   b. forward pass
   c. back propagation
   d. parameters update
4. Model Evaluation


'''

In [8]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)

Train Test Split

In [12]:
X_train, X_test, y_train, y_test= train_test_split(df.iloc[:,1:], df.iloc[:,0], test_size=0.2)

Scaling

In [13]:
scaler= StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

Label encoding

In [14]:
encoder= LabelEncoder()
y_train= encoder.fit_transform(y_train)
y_test= encoder.transform(y_test)

In [15]:
y_test

array([1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0])

Numpy arrays to PyTorch tensors

In [16]:
X_train_tensor= torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor= torch.from_numpy(y_train)
y_test_tensor= torch.from_numpy(y_test)

In [17]:
X_train_tensor

tensor([[-1.1863, -0.3845, -1.1625,  ..., -0.8043, -0.0060, -0.4062],
        [ 0.6825,  0.2373,  0.5796,  ..., -0.4577,  2.9155, -0.4291],
        [-1.3700,  0.5819, -1.3527,  ..., -1.4136, -0.8547,  0.4557],
        ...,
        [ 0.0211, -1.3300, -0.0158,  ..., -0.1275,  0.2693, -1.2960],
        [-0.9301, -0.3635, -0.9022,  ..., -0.2837, -0.9972, -0.0630],
        [-1.2631, -0.5103, -1.2405,  ..., -0.9847, -0.5974,  0.0432]],
       dtype=torch.float64)

Define the Model

In [27]:
class MySimpleNN():
    def __init__(self, X):
        self.weights= torch.rand(X.shape[1],1,dtype= torch.float64, requires_grad= True)

        self.bias= torch.zeros(1,dtype= torch.float64, requires_grad= True)
    
    def forward(self,X):
        z= torch.matmul(X,self.weights )+ self.bias
        y_pred= torch.sigmoid(z)

        return y_pred
    
    def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
       epsilon = 1e-7
       y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
       loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
       return loss

Important parameters

In [20]:
learning_rate= 0.1
epochs=25

Training Pipeline

In [31]:
#Create Model

model= MySimpleNN(X_train_tensor)

# define loop
for epoch in range(epochs):
  #forward pass
  y_pred= model.forward(X_train_tensor)

  # loss calucation
  loss= model.loss_function(y_pred,y_train_tensor)
  
  # backpropagation
  
  loss.backward()

  #parameters update
  with torch.no_grad():
    model.weights-=learning_rate*model.weights.grad
    model.bias-=learning_rate*model.bias.grad
  
  #zero gradients
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  #print loss in each epoch
  print(f"epoch: {epoch+1}, Loss: {loss.item()}")


epoch: 1, Loss: 3.559823661492285
epoch: 2, Loss: 3.4302815502742097
epoch: 3, Loss: 3.294359651505682
epoch: 4, Loss: 3.15433547053553
epoch: 5, Loss: 3.0068773860981985
epoch: 6, Loss: 2.855726997122032
epoch: 7, Loss: 2.703187563973577
epoch: 8, Loss: 2.545127387010487
epoch: 9, Loss: 2.381614331442613
epoch: 10, Loss: 2.209323268591382
epoch: 11, Loss: 2.0374553276718905
epoch: 12, Loss: 1.8654929237838753
epoch: 13, Loss: 1.6922989301646765
epoch: 14, Loss: 1.5228557200514021
epoch: 15, Loss: 1.3632969175081753
epoch: 16, Loss: 1.21989517110862
epoch: 17, Loss: 1.0955512596393469
epoch: 18, Loss: 0.992442563170567
epoch: 19, Loss: 0.9112992303011707
epoch: 20, Loss: 0.8509548879302414
epoch: 21, Loss: 0.8084934086904427
epoch: 22, Loss: 0.7799614147187494
epoch: 23, Loss: 0.7612875360011617
epoch: 24, Loss: 0.749030129535455
epoch: 25, Loss: 0.7407104553948753


In [32]:
model.weights

tensor([[ 0.3777],
        [-0.0774],
        [ 0.1183],
        [ 0.2226],
        [-0.2360],
        [-0.0643],
        [-0.1632],
        [-0.2876],
        [-0.0665],
        [ 0.1489],
        [-0.2939],
        [-0.0517],
        [ 0.5417],
        [-0.1295],
        [ 0.3650],
        [ 0.0598],
        [-0.2422],
        [ 0.3673],
        [ 0.2270],
        [-0.1719],
        [-0.0996],
        [ 0.0146],
        [-0.0080],
        [-0.3379],
        [ 0.0408],
        [ 0.3624],
        [-0.0541],
        [-0.0355],
        [ 0.4538],
        [-0.1439]], dtype=torch.float64, requires_grad=True)

Evaluation


In [35]:
#Model Evaluation
with torch.no_grad():
    y_pred= model.forward(X_test_tensor)
    y_pred= (y_pred>0.5).float()
    accuracy= (y_pred== y_test_tensor).float().mean()
    print(f"Accuracy: {accuracy.item()}")
print(y_pred)

Accuracy: 0.55755615234375
tensor([[0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
       